# Collect Data

In [1]:
%set_env SPEECH_KEY=3aa7bc5801d04e9286a201acc344cb13
%set_env SPEECH_REGION=westus

import os
import azure.cognitiveservices.speech as speechsdk

# This example requires environment variables named "SPEECH_KEY" and "SPEECH_REGION"
speech_config = speechsdk.SpeechConfig(subscription=os.environ.get('SPEECH_KEY'), region=os.environ.get('SPEECH_REGION'))
audio_config = speechsdk.audio.AudioOutputConfig(use_default_speaker=True)

# The neural multilingual voice can speak different languages based on the input text.
speech_config.speech_synthesis_voice_name='en-US-AvaMultilingualNeural'

speech_synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)

def say_it(text = "Speech system test"):

    speech_synthesis_result = speech_synthesizer.speak_text_async(text).get()

    if speech_synthesis_result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
        print("Speech synthesized for text [{}]".format(text))
    elif speech_synthesis_result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = speech_synthesis_result.cancellation_details
        print("Speech synthesis canceled: {}".format(cancellation_details.reason))
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            if cancellation_details.error_details:
                print("Error details: {}".format(cancellation_details.error_details))
                print("Did you set the speech resource key and region values?")

say_it()

env: SPEECH_KEY=3aa7bc5801d04e9286a201acc344cb13
env: SPEECH_REGION=westus
Speech synthesized for text [Speech system test]


In [2]:
import threading
import time
import datetime

import shutil

import pyautogui

from pynput.keyboard import Key, Listener
from pynput import mouse 
import logging
import os
from win32gui import GetWindowText, GetForegroundWindow


session_key = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

#### TODO: override
session_key = "2024-09-04_12-36-00"

output_dir = f"output/session_{session_key}"
print("output_dir", output_dir)

os.makedirs(output_dir, exist_ok=True)

images_dir = f"{output_dir}/images"
os.makedirs(images_dir, exist_ok=True)

# Set up logging to log keystrokes to a file
#logging.basicConfig(filename=(f"{output_dir}/keylog.txt"), level=logging.DEBUG, format="%(message)s")

out_file = open(f"{output_dir}/keylog.txt", "a")

def capture_screenshot(out_dir, key):
    # Capture the screenshot
    screenshot = pyautogui.screenshot()

    file_name = f"screenshot_{key}"
    output_file = f"{out_dir}/{file_name}.png"
    # Save the screenshot
    screenshot.save(output_file)
    return file_name, output_file








output_dir output/session_2024-09-04_12-36-00


In [3]:
import winsound
import json

# ==========================================================
class UserEventRecorder:
    def __init__(self, 
                 activity_timeout = 7, 
                 activity_grace_timeout = 1,
                 screen_poll_interval = 1,
                 target_app = "Microsoft​ Edge") -> None:
        
        self.last_activity_ts = datetime.datetime.now()
        
        self.activity_timeout = activity_timeout
        self.activity_grace_timeout = activity_grace_timeout
        self.screen_poll_interval = screen_poll_interval
        self.target_app = target_app
        self.command_buffer = []
        self.timeline_screenshots = []


    def update_command_buffer(self, s):
        if s:
            action_buffer = self.command_buffer
            
            if len(s) == 1:
                if len(action_buffer) > 0:
                    prev_action = action_buffer[-1]
                    if prev_action["type"] == "text":
                        prev_action["text"] = prev_action["text"] + s
                        action_buffer[-1] = prev_action
                    else:
                        action_buffer.append({"type" : "text", "text" : s})            
            else:
                # min_buffer_sz = 2
                # if "image" == s["type"] and len(action_buffer) > min_buffer_sz: 
                #     logging.info(f"{action_buffer[:-min_buffer_sz]}\t{action_buffer[-min_buffer_sz:]}")
                
                action_buffer.append(s)


    def get_current_app(self):
        return GetWindowText(GetForegroundWindow())

    def find_preceeding_screenshot(self, ts_now):
        print("finding preceeding...", len(self.timeline_screenshots))

        for i in range(len(self.timeline_screenshots) - 1, 0, -1):
            print("....................................")
            timeline = self.timeline_screenshots[i]
            s_ts = timeline[0]
            fn = timeline[1]

            print(f"s_ts: {s_ts}, {ts_now}")
            if s_ts < ts_now:
                print("Found preceeding: ", fn)
                return fn
            
        return None



    def record_user_action(self, action):
        
        
        if self.target_app in self.get_current_app():

            ts_now = datetime.datetime.now()

            time_difference = ts_now - self.last_activity_ts
            seconds = time_difference.total_seconds()
            
            if seconds > 1:
                curren_buffer = self.get_current_activity_buffer()
                if len(curren_buffer) > 0:
                    prev_action = curren_buffer[-1]
                    if prev_action["type"] == "sleep":
                        curren_buffer[-1] = {"type" : "sleep", "seconds" : seconds}
                    else:
                        self.update_command_buffer({"type" : "sleep", "seconds" : seconds})
                else:
                    self.update_command_buffer({"type" : "sleep", "seconds" : seconds})

            preceeding_screen = self.find_preceeding_screenshot(ts_now)
            print("Preceeding screen:", preceeding_screen)
            if preceeding_screen:
                if seconds > self.activity_grace_timeout:

                    self.update_command_buffer({"type" : "image", "path" : preceeding_screen})                
                
            self.update_command_buffer(action)
            self.last_activity_ts = ts_now

    def on_move(self, x, y):
        #self.update_last_activity_ts()
        # print('Pointer moved to {0}'.format(
        #     (x, y)))
        #update_command_buffer(f" <mouse>on_move({x},{y})")
        pass

    def on_click(self, x, y, button, pressed):
        if pressed:
            print("<event> Mouse Pressed")
            #self.record_user_action(f"<mouse>on_click({x},{y},{button},{pressed})")
            self.record_user_action({"type" : "mouse", "x" : x, "y" : y})

    def on_scroll(self, x, y, dx, dy):
        #self.update_last_activity_ts()
        # print('Scrolled {0} at {1}'.format(
        #     'down' if dy < 0 else 'up',
        #     (x, y)))
        #update_command_buffer(f" <mouse>on_scroll({x},{y},{dx},{dy})")
        pass


    def process_on_key(self, key, pressed):
        s = f"{key}"
        if len(s) > 0 and s[0] == '\'' and s[-1] == '\'':
            s = s[1:-1]

        tag = ""
        if "Key." in s:
            tag = "key"   
            s = {"type" : tag, "name" : s}
        
        self.record_user_action(s)
        #pass
        

    def on_press(self, key):

        s = self.process_on_key(key, True)         
        self.record_user_action(s)
        # self.update_command_buffer(s)
        pass

    def on_release(self, key):

        #s = self.process_on_key(key, False)
        #self.record_user_action(s)        
        # self.update_command_buffer(s)
        pass

    def get_current_activity_buffer(self):
        return self.command_buffer

    def record_screen(self):
        ts_now = datetime.datetime.now()                
        ts = ts_now.strftime('%Y-%m-%d_%H-%M-%S.%f')
        
        event_screenshot, _ = capture_screenshot(images_dir, ts)
        self.timeline_screenshots.append([ts_now, event_screenshot])


    def start_new_activity(self):
        self.command_buffer = []
        self.timeline_screenshots = []

        self.record_screen()        
        self.update_command_buffer({"type" : "image", "path" : self.timeline_screenshots[0][1]})


    def heartbeat(self):

        def beep():
            frequency = 500  # Set Frequency To 2500 Hertz
            duration = 777  # Set Duration To 1000 ms == 1 second
            winsound.Beep(frequency, duration)

        activity_started = False

        while True:

            #print("Current app:", self.get_current_app())
            current_app = self.get_current_app()

            if self.target_app in current_app:
                if not activity_started:
                    say_it("Starting new activity recording.")
                    self.start_new_activity()
                    activity_started = True

                beep()
                self.record_screen()
                
            else:
                if activity_started:
                    activity_started = False
                    say_it("Stopping activity recording")
                    print("Stopped for app:", current_app)
                    if len(self.command_buffer) > 0:
                        print(">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
                        print(self.command_buffer)
                        #logging.info(json.dumps(self.command_buffer))
                        out_file.write(json.dumps(self.command_buffer))
                        out_file.write("\n")
                        out_file.flush()
                        print("<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<")
                    else:
                        print("???????????????????????????????")
                        print("Empty Buffer")
            time.sleep(self.screen_poll_interval)

    def start_key_listener(self):
        # Start listening to keystrokes
        with Listener(on_press=self.on_press, on_release=self.on_release) as listener:
            listener.join()

In [4]:
        
user_events_recorder = UserEventRecorder()

heartbeat_thread = threading.Thread(target = user_events_recorder.heartbeat)

heartbeat_thread.start()

key_listener_thread = threading.Thread(target = user_events_recorder.start_key_listener)

key_listener_thread.start()

# Collect events until released
with mouse.Listener(
        on_move = user_events_recorder.on_move,
        on_click = user_events_recorder.on_click,
        on_scroll = user_events_recorder.on_scroll) as listener:
    listener.join()



Speech synthesized for text [Starting new activity recording.]
<event> Mouse Pressed
finding preceeding... 2
....................................
s_ts: 2024-09-04 14:27:56.585298, 2024-09-04 14:27:57.661452
Found preceeding:  screenshot_2024-09-04_14-27-56.585298
Preceeding screen: screenshot_2024-09-04_14-27-56.585298
finding preceeding... 2
....................................
s_ts: 2024-09-04 14:27:56.585298, 2024-09-04 14:27:58.912414
Found preceeding:  screenshot_2024-09-04_14-27-56.585298
Preceeding screen: screenshot_2024-09-04_14-27-56.585298
finding preceeding... 2
....................................
s_ts: 2024-09-04 14:27:56.585298, 2024-09-04 14:27:58.919075
Found preceeding:  screenshot_2024-09-04_14-27-56.585298
Preceeding screen: screenshot_2024-09-04_14-27-56.585298
finding preceeding... 2
....................................
s_ts: 2024-09-04 14:27:56.585298, 2024-09-04 14:27:59.140658
Found preceeding:  screenshot_2024-09-04_14-27-56.585298
Preceeding screen: screensho